In [2]:
import sys, json
sys.path.append("..")
import numpy as np
import pandas as pd
from src.baseline import baseline_models
from src.ablation import fold_scores

X_train = pd.read_parquet("../data/processed/X_train.parquet")
y_train = pd.read_parquet("../data/processed/y_train.parquet")["_MICHD"]
sel = json.load(open("../reports/selected_features.json"))
full, minimal = sel["full"], sel["minimal"]

no_emp = lambda feats: [f for f in feats if f != "EMPLOY1"]

# derive=True is switched off automatically whenever the 7 condition columns
# aren't all present, so chronic_count only ever appears in A and C
variants = {
    "A full29 + chronic_count (reference)": (full, True),
    "B full29, no chronic_count":           (full, False),
    "C full29 - EMPLOY1 (+ chronic_count)": (no_emp(full), True),
    "D minimal15":                          (minimal, True),
    "E minimal15 - EMPLOY1":                (no_emp(minimal), True),
    "F minimal10 (top 10 by rank)":         (minimal[:10], True),
}
print({k: len(v[0]) for k, v in variants.items()})

{'A full29 + chronic_count (reference)': 29, 'B full29, no chronic_count': 29, 'C full29 - EMPLOY1 (+ chronic_count)': 28, 'D minimal15': 15, 'E minimal15 - EMPLOY1': 14, 'F minimal10 (top 10 by rank)': 10}


In [3]:
rows = []
for model_name in ["logreg", "xgb"]:
    ref = None
    for vname, (feats, derive) in variants.items():
        s = fold_scores(baseline_models()[model_name], feats, X_train, y_train, derive)
        ref = s if ref is None else ref
        d_roc = s["roc_auc"] - ref["roc_auc"]
        d_pr = s["pr_auc"] - ref["pr_auc"]
        rows.append({"model": model_name, "variant": vname, "n_raw_features": len(feats),
                     "roc_auc": s["roc_auc"].mean(), "pr_auc": s["pr_auc"].mean(),
                     "d_roc_vs_A": d_roc.mean(), "d_roc_spread": d_roc.std(),
                     "d_pr_vs_A": d_pr.mean(), "d_pr_spread": d_pr.std()})
        print(f"{model_name:7s} {vname:40s} ROC-AUC={s['roc_auc'].mean():.4f}")

abl = pd.DataFrame(rows)

logreg  A full29 + chronic_count (reference)     ROC-AUC=0.8295
logreg  B full29, no chronic_count               ROC-AUC=0.8295
logreg  C full29 - EMPLOY1 (+ chronic_count)     ROC-AUC=0.8289
logreg  D minimal15                              ROC-AUC=0.8277
logreg  E minimal15 - EMPLOY1                    ROC-AUC=0.8269
logreg  F minimal10 (top 10 by rank)             ROC-AUC=0.8253
xgb     A full29 + chronic_count (reference)     ROC-AUC=0.8334
xgb     B full29, no chronic_count               ROC-AUC=0.8333
xgb     C full29 - EMPLOY1 (+ chronic_count)     ROC-AUC=0.8329
xgb     D minimal15                              ROC-AUC=0.8308
xgb     E minimal15 - EMPLOY1                    ROC-AUC=0.8302
xgb     F minimal10 (top 10 by rank)             ROC-AUC=0.8279


In [4]:
import os
os.makedirs("../reports", exist_ok=True)
abl.round(4).to_csv("../reports/ablation_results.csv", index=False)
abl.round(4)

,model,variant,n_raw_features,roc_auc,pr_auc,d_roc_vs_A,d_roc_spread,d_pr_vs_A,d_pr_spread
0,logreg,A full29 + chronic_count (reference),29,0.8295,0.3410,0.0000,0.0000,0.0000,0.0000
1,logreg,"B full29, no chronic_count",29,0.8295,0.3409,-0.0000,0.0000,-0.0001,0.0001
2,logreg,C full29 - EMPLOY1 (+ chronic_count),28,0.8289,0.3400,-0.0007,0.0001,-0.0010,0.0003
3,logreg,D minimal15,15,0.8277,0.3369,-0.0018,0.0004,-0.0041,0.0004
4,logreg,E minimal15 - EMPLOY1,14,0.8269,0.3357,-0.0026,0.0003,-0.0053,0.0003
5,logreg,F minimal10 (top 10 by rank),10,0.8253,0.3339,-0.0042,0.0006,-0.0071,0.0009
6,xgb,A full29 + chronic_count (reference),29,0.8334,0.3463,0.0000,0.0000,0.0000,0.0000
7,xgb,"B full29, no chronic_count",29,0.8333,0.3463,-0.0001,0.0003,-0.0000,0.0004
8,xgb,C full29 - EMPLOY1 (+ chronic_count),28,0.8329,0.3456,-0.0004,0.0002,-0.0007,0.0009
9,xgb,D minimal15,15,0.8308,0.3411,-0.0025,0.0005,-0.0052,0.0012


In [5]:
final = [f for f in minimal if f != "EMPLOY1"]
print(len(final), final)       # expect 14

json.dump({"final": final, "chronic_count": False, "benchmark_full": full},
          open("../reports/final_features.json", "w"), indent=2)

14 ['_AGE_G', 'GENHLTH', 'DIABETE4', 'CHCCOPD3', 'DIFFWALK', 'HAVARTH4', 'CHCKDNY2', 'MARITAL', 'SEXVAR', 'CHECKUP1', '_SMOKER3', 'PHYSHLTH', 'CHCOCNC1', 'INCOME3']


## Ablation findings

Variants were scored on identical CV folds with logistic regression and XGBoost (full training set), so per-fold differences are paired.

**Derived feature.** Adding `chronic_count` (sum of seven condition flags) changed ROC-AUC by ≤ 0.0001 for both models: it is redundant with the columns it is built from, and was dropped.

**Employment status.** Despite ranking 3rd in the multi-method selection, removing `EMPLOY1` cost only 0.0004–0.0008 ROC-AUC (consistent across folds, but practically negligible). Its signal overlaps with age, difficulty walking and general health, so marginal importance overstated its unique contribution. It was dropped.

**Number of questions.** Reducing from 29 to 15 questions cost about 0.002–0.003 ROC-AUC and 0.004–0.005 PR-AUC; reducing further to 10 cost an additional 0.002–0.003 ROC-AUC. A 14-question final set was chosen as the point before the loss becomes practically meaningful.

**Final feature set (14):** age group, general health, diabetes status, COPD, difficulty walking, arthritis, kidney disease, marital status, sex, time since last checkup, smoking status, days of poor physical health, skin-cancer history (other cancer), and income.